In [4]:
!python -m pip install opencv-python

  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   - -------------------------------------- 1.0/40.2 MB 8.9 MB/s eta 0:00:05
   --- ------------------------------------ 3.7/40.2 MB 10.9 MB/s eta 0:00:04
   ------ --------------------------------- 6.3/40.2 MB 11.4 MB/s eta 0:00:03
   ---------- ----------------------------- 10.2/40.2 MB 13.6 MB/s eta 0:00:03
   -------------- ------------------------- 14.7/40.2 MB 15.3 MB/s eta 0:00:02
   ------------------- -------------------- 19.4/40.2 MB 16.5 MB/s eta 0:00:02
   ----------------------- ---------------- 23.3/40.2 MB 16.9 MB/s eta 0:00:02
   ------------------------------ --------- 30.4/40.2 MB 19.1 MB/s eta 0:00:01
   ------------------------------------ --- 36.4/40.2 MB 20.2 MB/s eta 0:00:01
   ---------------------------------------  40.1/40.2 MB 20.7 MB/s eta 0:00:01
   ---------------------------------------- 40.2/40.2 MB 18.8 MB


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import cv2
import torch
import numpy as np
import torch.nn as nn
import time
from collections import deque
from torchvision.models.video import r3d_18, R3D_18_Weights

# -------------------------------
# DEVICE
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------
# LOAD FEATURE EXTRACTOR
# -------------------------------
weights = R3D_18_Weights.DEFAULT
r3d = r3d_18(weights=weights)
r3d.fc = nn.Identity()
r3d = r3d.to(device).eval()

# -------------------------------
# MODEL
# -------------------------------
class Attention(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.attn = nn.Linear(input_dim, 1)

    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        return torch.sum(x * weights, dim=1)

class CrimeBiLSTM_Attention(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=256, num_classes=14):
        super().__init__()

        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.attention = Attention(hidden_dim * 2)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        context = self.attention(lstm_out)
        return self.classifier(context)

# -------------------------------
# LOAD MODEL
# -------------------------------
label_map = np.load("LSTM/bilstm_label_map.npy", allow_pickle=True).item()
idx_to_label = {v: k for k, v in label_map.items()}

model = CrimeBiLSTM_Attention(num_classes=len(label_map))
model.load_state_dict(torch.load("LSTM/bilstm_attention_model.pth", map_location=device))
model = model.to(device).eval()

# -------------------------------
# NORMALIZATION
# -------------------------------
mean = torch.tensor([0.43216, 0.394666, 0.37645]).view(1,3,1,1,1).to(device)
std = torch.tensor([0.22803, 0.22145, 0.216989]).view(1,3,1,1,1).to(device)

# -------------------------------
# SETTINGS
# -------------------------------
clip_len = 16
segments = 8
CONF_THRESHOLD = 0.6
COOLDOWN = 5  # seconds

buffer = []
prediction_queue = deque(maxlen=5)

last_logged_time = 0
LOG_FILE = "crime_log.txt"

cap = cv2.VideoCapture(0)

print("Press ESC to exit")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    display_frame = frame.copy()

    frame = cv2.resize(frame, (112, 112))
    buffer.append(frame)

    if len(buffer) >= clip_len * segments:

        indices = np.linspace(0, len(buffer) - clip_len, segments).astype(int)

        clips = []
        for idx in indices:
            clip = buffer[idx:idx + clip_len]
            clip = torch.from_numpy(np.array(clip)).permute(3,0,1,2).float()/255.0
            clips.append(clip)

        clips = torch.stack(clips).to(device)
        clips = (clips - mean) / std

        # FEATURE EXTRACTION
        with torch.no_grad():
            features = r3d(clips)

        features = features.unsqueeze(0)

        # MODEL PREDICTION
        with torch.no_grad():
            output = model(features)
            probs = torch.softmax(output, dim=1)

        pred = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred].item()
        label = idx_to_label[pred]

        # TEMPORAL SMOOTHING
        prediction_queue.append(label)
        final_label = max(set(prediction_queue), key=prediction_queue.count)

        # DISPLAY
        cv2.putText(display_frame, f"{final_label} ({confidence:.2f})",
                    (20,40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

        # ALERT + LOGGING
        current_time = time.time()

        if final_label != "Normal" and confidence > CONF_THRESHOLD:
            if current_time - last_logged_time > COOLDOWN:

                last_logged_time = current_time

                timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
                log_entry = f"{timestamp} | Crime: {final_label} | Confidence: {confidence:.2f}\n"

                with open(LOG_FILE, "a") as f:
                    f.write(log_entry)

                print("⚠️ ALERT:", log_entry)

                cv2.putText(display_frame, "ALERT!", (20,80),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 3)

        buffer = []

    cv2.imshow("Crime Detection", display_frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

Press ESC to exit
⚠️ ALERT: 2026-05-03 14:28:54 | Crime: NormalVideos | Confidence: 0.95

⚠️ ALERT: 2026-05-03 14:29:05 | Crime: NormalVideos | Confidence: 0.97

⚠️ ALERT: 2026-05-03 14:29:18 | Crime: NormalVideos | Confidence: 0.78

⚠️ ALERT: 2026-05-03 14:29:29 | Crime: NormalVideos | Confidence: 0.84

⚠️ ALERT: 2026-05-03 14:29:40 | Crime: NormalVideos | Confidence: 0.91



In [ ]:
import cv2
import torch
import numpy as np
import torch.nn as nn
import time
from collections import deque
from torchvision.models.video import r3d_18, R3D_18_Weights


# DEVICE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# LOAD FEATURE EXTRACTOR

weights = R3D_18_Weights.DEFAULT
r3d = r3d_18(weights=weights)
r3d.fc = nn.Identity()
r3d = r3d.to(device).eval()


# MODEL (UNCHANGED)

class Attention(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.attn = nn.Linear(input_dim, 1)

    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        return torch.sum(x * weights, dim=1)

class CrimeBiLSTM_Attention(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=256, num_classes=14):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.attention = Attention(hidden_dim * 2)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        context = self.attention(lstm_out)
        return self.classifier(context)


# LOAD MODEL + LABEL MAP

label_map = np.load("LSTM/bilstm_label_map.npy", allow_pickle=True).item()
idx_to_label = {v: k for k, v in label_map.items()}

model = CrimeBiLSTM_Attention(num_classes=len(label_map))
model.load_state_dict(torch.load("LSTM/bilstm_attention_model.pth", map_location=device))
model = model.to(device).eval()


# NORMALIZATION

mean = torch.tensor([0.43216, 0.394666, 0.37645]).view(1,3,1,1,1).to(device)
std = torch.tensor([0.22803, 0.22145, 0.216989]).view(1,3,1,1,1).to(device)


# OPTIMIZED SETTINGS

clip_len = 8
segments = 4
frame_skip = 2

CONF_THRESHOLD = 0.6
COOLDOWN = 5

buffer = deque(maxlen=clip_len * segments)
prediction_queue = deque(maxlen=5)

last_logged_time = 0
frame_count = 0
last_prediction = "Initializing..."

LOG_FILE = "crime_log.txt"


# WEBCAM

cap = cv2.VideoCapture(0)

print("Press ESC to exit")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    display_frame = frame.copy()

    # FRAME SKIPPING
    frame_count += 1
    if frame_count % frame_skip != 0:
        cv2.putText(display_frame, last_prediction, (20,40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
        cv2.imshow("Crime Detection", display_frame)

        if cv2.waitKey(1) & 0xFF == 27:
            break
        continue

    # PREPROCESS
    frame_small = cv2.resize(frame, (112, 112))
    buffer.append(frame_small)

    # INFERENCE
    if len(buffer) >= clip_len * segments:

        indices = np.linspace(0, len(buffer) - clip_len, segments).astype(int)

        clips = []
        for idx in indices:
            clip = list(buffer)[idx:idx + clip_len]
            clip = torch.from_numpy(np.array(clip)).permute(3,0,1,2).float()/255.0
            clips.append(clip)

        clips = torch.stack(clips).to(device)
        clips = (clips - mean) / std

        with torch.no_grad():
            features = r3d(clips)

        features = features.unsqueeze(0)

        with torch.no_grad():
            output = model(features)
            probs = torch.softmax(output, dim=1)

        pred = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred].item()
        label = idx_to_label[pred]

            # SMOOTHING
            prediction_queue.append(label)
        final_label = max(set(prediction_queue), key=prediction_queue.count)

        last_prediction = f"{final_label} ({confidence:.2f})"

            # LOGGING
            current_time = time.time()

        if final_label != "Normal" and confidence > CONF_THRESHOLD:
            if current_time - last_logged_time > COOLDOWN:

                last_logged_time = current_time

                timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
                log_entry = f"{timestamp} | Crime: {final_label} | Confidence: {confidence:.2f}\n"

                with open(LOG_FILE, "a") as f:
                    f.write(log_entry)

                print("⚠️ ALERT:", log_entry)

                cv2.putText(display_frame, "ALERT!", (20,80),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 3)

    # DISPLAY
    cv2.putText(display_frame, last_prediction, (20,40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    cv2.imshow("Crime Detection", display_frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

Press ESC to exit
⚠️ ALERT: 2026-05-03 14:40:39 | Crime: Abuse | Confidence: 0.72

⚠️ ALERT: 2026-05-03 14:40:44 | Crime: Abuse | Confidence: 0.65

⚠️ ALERT: 2026-05-03 14:40:50 | Crime: Abuse | Confidence: 0.67

⚠️ ALERT: 2026-05-03 14:40:56 | Crime: Abuse | Confidence: 0.85

⚠️ ALERT: 2026-05-03 14:41:03 | Crime: Abuse | Confidence: 0.75

⚠️ ALERT: 2026-05-03 14:41:09 | Crime: Abuse | Confidence: 0.80

⚠️ ALERT: 2026-05-03 14:41:16 | Crime: Abuse | Confidence: 0.78

⚠️ ALERT: 2026-05-03 14:41:22 | Crime: Abuse | Confidence: 0.86

